In [2]:
import sys, os
while not os.path.isdir('src') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')
sys.path.insert(0, 'src')

import numpy as np, json
import matplotlib.pyplot as plt
import torch
os.makedirs('dashboard/data', exist_ok=True)

from tradinglab.data_feed import DataFeed
from tradinglab.features import build_pooled_dataset, build_pooled_sequences

EPOCHS = 150
TOP_K = 4
SEQ_LEN = 10
HIDDEN = 32

feed = DataFeed.from_dir('data/egx')   # full universe -- every symbol in data/egx
print('universe:', feed.n_assets, 'symbols |', feed.n_days, 'trading days')

split_day = int(feed.n_days * 0.7)
X_train, y_train, X_test, y_test = build_pooled_dataset(feed, split_day)
Xs_train, ys_train, Xs_test, ys_test = build_pooled_sequences(feed, split_day, seq_len=SEQ_LEN)
n_features = X_train.shape[1]

# the RuntimeWarning above comes from a column that's computed but never used --
# confirm it doesn't leak into the actual training data
assert not np.isnan(X_train).any() and not np.isnan(X_test).any()
assert not np.isnan(Xs_train).any() and not np.isnan(Xs_test).any()

print('flat      :', len(X_train), 'train /', len(X_test), 'test |', n_features, 'features')
print('sequences :', Xs_train.shape, 'train /', Xs_test.shape, 'test')

universe: 34 symbols | 1159 trading days


/Users/judyhedayah/Desktop/2026-orion-1/src/tradinglab/features.py:64: RuntimeWarning: invalid value encountered in divide
  volume_ratio = volume / vol_avg20


flat      : 26588 train / 11798 test | 5 features
sequences : (26248, 10, 5) train / (11458, 10, 5) test


In [3]:
torch.manual_seed(0)
from tradinglab.models import MLP
from tradinglab.ml import train_model, predict

mlp = MLP(n_features, hidden=HIDDEN)
mlp_history = train_model(mlp, X_train, y_train, X_test, y_test, epochs=EPOCHS)

plt.figure(figsize=(9, 4))
plt.plot(mlp_history['train'], label='train loss')
plt.plot(mlp_history['test'], label='test loss')
plt.yscale('log')
plt.legend(); plt.grid(alpha=.3); plt.title('MLP -- watch the gap = overfitting')
plt.show()
print('final train %.5f | test %.5f' % (mlp_history['train'][-1], mlp_history['test'][-1]))

AttributeError: module 'torch' has no attribute 'manual_seed'

In [ ]:
torch.manual_seed(0)
from tradinglab.models import LSTMRegressor

lstm = LSTMRegressor(n_features=n_features, hidden=HIDDEN)
lstm_history = train_model(lstm, Xs_train, ys_train, Xs_test, ys_test, epochs=EPOCHS)

plt.figure(figsize=(9, 4))
plt.plot(lstm_history['train'], label='train loss')
plt.plot(lstm_history['test'], label='test loss')
plt.yscale('log')
plt.legend(); plt.grid(alpha=.3); plt.title('LSTM -- watch the gap = overfitting')
plt.show()
print('final train %.5f | test %.5f' % (lstm_history['train'][-1], lstm_history['test'][-1]))

In [ ]:
from tradinglab.strategies.predictor import predictions_to_weights, model_to_strategy

def lstm_to_strategy(model, seq_len, top_k=2):
    def strategy(observation):
        window = observation[:, -seq_len:, :]              # take only the last seq_len days
        preds = predict(model, window.astype('float32'))
        return predictions_to_weights(preds, top_k)
    return strategy

mlp_strategy = model_to_strategy(mlp, top_k=TOP_K)
lstm_strategy = lstm_to_strategy(lstm, seq_len=SEQ_LEN, top_k=TOP_K)

In [ ]:
from tradinglab.simulator import PortfolioSimulator
from tradinglab.backtester import run_backtest
from tradinglab.report import report

sim = PortfolioSimulator(feed)
split = split_day

result_mlp = run_backtest(sim, mlp_strategy, lookback=30, start=split)
report(result_mlp, title='MLP strategy vs benchmark -- full universe (test period)', start_capital=1000.0)

json.dump({'portfolio': [round(x, 4) for x in result_mlp['portfolio'].tolist()],
           'benchmark': [round(x, 4) for x in result_mlp['benchmark'].tolist()]},
          open('dashboard/data/day3_mlp_equity.json', 'w'))

In [ ]:
result_lstm = run_backtest(sim, lstm_strategy, lookback=30, start=split)
report(result_lstm, title='LSTM strategy vs benchmark -- full universe (test period)', start_capital=1000.0)

json.dump({'portfolio': [round(x, 4) for x in result_lstm['portfolio'].tolist()],
           'benchmark': [round(x, 4) for x in result_lstm['benchmark'].tolist()]},
          open('dashboard/data/day3_lstm_equity.json', 'w'))

In [ ]:
START = 1000.0
dates = result_mlp['dates']

plt.figure(figsize=(11, 5))
plt.plot(dates, result_mlp['portfolio'] * START, label='MLP')
plt.plot(dates, result_lstm['portfolio'] * START, label='LSTM')
plt.plot(dates, result_mlp['benchmark'] * START, label='benchmark', linestyle='--')
plt.legend(); plt.grid(alpha=.3); plt.ylabel('EGP')
plt.title('MLP vs LSTM vs benchmark -- full universe (test period)')
plt.gcf().autofmt_xdate()
plt.show()

mlp_final = result_mlp['portfolio'][-1] * START
lstm_final = result_lstm['portfolio'][-1] * START
bench_final = result_mlp['benchmark'][-1] * START

print(f'MLP final  : {mlp_final:12,.0f} EGP  -- beat benchmark: {"YES" if mlp_final > bench_final else "no"}')
print(f'LSTM final : {lstm_final:12,.0f} EGP  -- beat benchmark: {"YES" if lstm_final > bench_final else "no"}')
print(f'benchmark  : {bench_final:12,.0f} EGP')

winner = 'LSTM' if lstm_final > mlp_final else 'MLP'
print(f'\n{winner} beat the other model by {abs(lstm_final - mlp_final):,.0f} EGP.')

In [ ]:
HIDDEN_GRID = [16, 32]
SEQ_LEN_GRID = [5, 10, 20]
TOPK_GRID = [3, 5]
SWEEP_EPOCHS = 60

seq_cache = {}
def get_sequences(seq_len):
    if seq_len not in seq_cache:
        seq_cache[seq_len] = build_pooled_sequences(feed, split_day, seq_len=seq_len)
    return seq_cache[seq_len]

sweep_rows = []
print(f'{"hidden":>6s} {"seq_len":>7s} {"top_k":>5s} {"final equity (EGP)":>19s}')
for seq_len in SEQ_LEN_GRID:
    Xtr_sw, ytr_sw, Xte_sw, yte_sw = get_sequences(seq_len)
    for hidden in HIDDEN_GRID:
        torch.manual_seed(0)
        m = LSTMRegressor(n_features=Xtr_sw.shape[2], hidden=hidden)
        train_model(m, Xtr_sw, ytr_sw, Xte_sw, yte_sw, epochs=SWEEP_EPOCHS)
        for top_k in TOPK_GRID:
            strat = lstm_to_strategy(m, seq_len=seq_len, top_k=top_k)
            res = run_backtest(sim, strat, lookback=30, start=split)
            final_equity = res['portfolio'][-1] * START
            sweep_rows.append({'hidden': hidden, 'seq_len': seq_len, 'top_k': top_k,
                                'final_equity': final_equity})
            print(f'{hidden:6d} {seq_len:7d} {top_k:5d} {final_equity:19,.0f}')

best = max(sweep_rows, key=lambda r: r['final_equity'])
print(f"\nbest config -> hidden={best['hidden']}  seq_len={best['seq_len']}  "
      f"top_k={best['top_k']}  final equity {best['final_equity']:,.0f} EGP")

In [ ]:
from tradinglab.metrics import information_coefficient

BEST_HIDDEN, BEST_SEQ_LEN, BEST_TOPK = best['hidden'], best['seq_len'], best['top_k']
Xtr_best, ytr_best, Xte_best, yte_best = get_sequences(BEST_SEQ_LEN)

print(f'{"seed":>5s} {"IC":>8s} {"strategy (EGP)":>15s} {"benchmark (EGP)":>16s}  beat?')
robust_rows = []
for seed in range(6):
    torch.manual_seed(seed)
    m = LSTMRegressor(n_features=Xtr_best.shape[2], hidden=BEST_HIDDEN)
    train_model(m, Xtr_best, ytr_best, Xte_best, yte_best, epochs=EPOCHS)
    seed_ic = information_coefficient(predict(m, Xte_best), yte_best)
    strat = lstm_to_strategy(m, seq_len=BEST_SEQ_LEN, top_k=BEST_TOPK)
    res = run_backtest(sim, strat, lookback=30, start=split)
    final, bench = res['portfolio'][-1] * START, res['benchmark'][-1] * START
    beat = final > bench
    robust_rows.append((seed, seed_ic, final, bench, beat))
    print(f'{seed:5d} {seed_ic:+8.3f} {final:15,.0f} {bench:16,.0f}  {"YES" if beat else "no"}')

wins = sum(r[4] for r in robust_rows)
print(f'\nwinning config beat the benchmark in {wins}/6 runs -- '
      f'same data, same architecture, only the seed changed.')

In [ ]:
def equal_weight_strategy(observation):
    n_assets = observation.shape[0]
    return np.ones(n_assets) / n_assets

result_equal_weight = run_backtest(sim, equal_weight_strategy, lookback=30, start=split)
print(f"Equal-weight benchmark final: {result_equal_weight['portfolio'][-1] * START:,.0f} EGP")

In [ ]:
# "My" models use ONLY raw daily returns as input (matching neural_network.ipynb
# and LSTM.ipynb exactly), not the professor's multi-feature indicator set.
# Pooled here across the FULL universe for a fair full-universe comparison
# (the originals only ever trained on a single stock, ABUK).

def build_pooled_raw_returns(feed, split_day, lookback):
    """NOTE: this is my own reproduction of pooling logic, not necessarily
    identical to tradinglab.features.build_pooled_dataset's internals — treat
    exact split boundaries as approximate, not guaranteed identical."""
    Xs, ys = [], []
    for a in range(feed.n_assets):
        r = feed.returns[:, a]
        for t in range(lookback, len(r) - 1):
            Xs.append(r[t - lookback:t])
            ys.append(r[t + 1])
    X = np.array(Xs, dtype='float32')
    y = np.array(ys, dtype='float32').reshape(-1, 1)
    split_idx = int(len(X) * (split_day / feed.n_days))
    return X[:split_idx], y[:split_idx], X[split_idx:], y[split_idx:]


# --- My MLP (neural_network.ipynb): input = yesterday's return only ---
MY_MLP_LOOKBACK = 1
X_my_mlp_train, y_my_mlp_train, X_my_mlp_test, y_my_mlp_test = build_pooled_raw_returns(feed, split_day, MY_MLP_LOOKBACK)

class MyMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 16), nn.ReLU(),
            nn.Linear(16, 8), nn.ReLU(),
            nn.Linear(8, 1),
        )
    def forward(self, x):
        return self.net(x)

torch.manual_seed(0)
my_mlp = MyMLP()
my_mlp_history = train_model(my_mlp, X_my_mlp_train, y_my_mlp_train, X_my_mlp_test, y_my_mlp_test, epochs=EPOCHS)
print(f"My MLP  final train {my_mlp_history['train'][-1]:.5f} | test {my_mlp_history['test'][-1]:.5f}")


# --- My LSTM (LSTM.ipynb): input = last 10 days of raw returns ---
MY_LSTM_LOOKBACK = 10
X_my_lstm_train, y_my_lstm_train, X_my_lstm_test, y_my_lstm_test = build_pooled_raw_returns(feed, split_day, MY_LSTM_LOOKBACK)

class MyLSTM(nn.Module):
    def __init__(self, hidden_size=16):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden_size, num_layers=1, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
    def forward(self, x):
        x_seq = x.unsqueeze(-1)
        out, _ = self.lstm(x_seq)
        return self.fc(out[:, -1, :])

torch.manual_seed(0)
my_lstm = MyLSTM()
my_lstm_history = train_model(my_lstm, X_my_lstm_train, y_my_lstm_train, X_my_lstm_test, y_my_lstm_test, epochs=EPOCHS)
print(f"My LSTM final train {my_lstm_history['train'][-1]:.5f} | test {my_lstm_history['test'][-1]:.5f}")

In [ ]:
def my_model_to_strategy(model, lookback, top_k=TOP_K):
    def strategy(observation):
        # observation: (n_assets, window, n_features) -- take only the raw
        # return column, last `lookback` days, per asset (my models only ever
        # saw returns, not the professor's other indicators)
        recent_returns = observation[:, -lookback:, 0].astype('float32')  # assumes returns is feature index 0
        model.eval()
        with torch.no_grad():
            x_t = torch.tensor(recent_returns)
            if lookback > 1:
                x_t = x_t  # MyLSTM does its own unsqueeze internally
            preds = model(x_t).squeeze(-1).numpy()
        return predictions_to_weights(preds, top_k)
    return strategy

my_mlp_strategy = my_model_to_strategy(my_mlp, lookback=MY_MLP_LOOKBACK, top_k=TOP_K)
my_lstm_strategy = my_model_to_strategy(my_lstm, lookback=MY_LSTM_LOOKBACK, top_k=TOP_K)

result_my_mlp = run_backtest(sim, my_mlp_strategy, lookback=30, start=split)
result_my_lstm = run_backtest(sim, my_lstm_strategy, lookback=30, start=split)

print(f"My MLP  final: {result_my_mlp['portfolio'][-1] * START:,.0f} EGP")
print(f"My LSTM final: {result_my_lstm['portfolio'][-1] * START:,.0f} EGP")

In [ ]:
plt.figure(figsize=(13, 6))
plt.plot(dates, result_my_mlp['portfolio'] * START, label='My MLP', linewidth=1.8, linestyle='-', color='#60a5fa')
plt.plot(dates, result_my_lstm['portfolio'] * START, label='My LSTM', linewidth=1.8, linestyle='-', color='#f59e0b')
plt.plot(dates, result_mlp['portfolio'] * START, label="Professor's MLP", linewidth=1.8, linestyle='--', color='#60a5fa')
plt.plot(dates, result_lstm['portfolio'] * START, label="Professor's LSTM", linewidth=1.8, linestyle='--', color='#f59e0b')
plt.plot(dates, result_equal_weight['portfolio'] * START, label='Equal-Weight Benchmark', linewidth=1.6, linestyle=':', color='#94a3b8')

plt.title('My models vs Professor\'s models vs Equal-Weight Benchmark')
plt.ylabel('Portfolio value (EGP)')
plt.legend()
plt.grid(alpha=0.3)
plt.gcf().autofmt_xdate()
plt.tight_layout()
plt.show()

finals = {
    'My MLP': result_my_mlp['portfolio'][-1] * START,
    'My LSTM': result_my_lstm['portfolio'][-1] * START,
    "Professor's MLP": result_mlp['portfolio'][-1] * START,
    "Professor's LSTM": result_lstm['portfolio'][-1] * START,
    'Equal-Weight Benchmark': result_equal_weight['portfolio'][-1] * START,
}
for name, val in sorted(finals.items(), key=lambda x: -x[1]):
    print(f"{name:22s}: {val:12,.0f} EGP")